# NB09 - Double Descent
## Background
Classical statistics predicts a U-shaped bias-variance tradeoff: as model complexity grows, test error first decreases then increases. Double descent (Belkin et al., 2019) shows a second descent after the interpolation threshold — once the model has enough capacity to perfectly fit the training data, further increasing capacity *reduces* test error again. This is counterintuitive: overparameterized models that memorize training data can still generalize.

The interpolation threshold is the point where model capacity equals training set size. Near this threshold (the 'double descent peak'), small models are in a capacity crisis: barely enough parameters to fit the data, so the solution is ill-conditioned and sensitive to noise.

## What You'll Learn
- Demonstrating the double descent curve on polynomial regression
- The interpolation threshold and why the peak occurs there
- How noise level affects the double descent peak height
- Connection to neural networks: epoch-wise double descent

## Why This Matters
Double descent explains why overparameterized neural networks generalize despite classical theory predicting they should not. It motivates using very large models with regularization rather than small models, and explains why training loss reaching zero is not a problem per se.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

# ── Double descent on polynomial regression ────────────────────────────────
np.random.seed(42)
n_train = 30
x_train = np.linspace(-1, 1, n_train)
y_train = np.sin(2*np.pi*x_train) + 0.3*np.random.randn(n_train)

n_test = 500
x_test = np.linspace(-1, 1, n_test)
y_test = np.sin(2*np.pi*x_test)

def fit_polynomial_ls(x_tr, y_tr, x_te, y_te, degree):
    """Fit polynomial via least squares (min-norm solution for overdetermined)."""
    X_tr = np.stack([x_tr**i for i in range(degree+1)], axis=-1)
    X_te = np.stack([x_te**i for i in range(degree+1)], axis=-1)
    # Minimum-norm least squares (pinv for overparameterized case)
    w = np.linalg.lstsq(X_tr, y_tr, rcond=None)[0]
    train_mse = np.mean((X_tr @ w - y_tr)**2)
    test_mse  = np.mean((X_te @ w - y_te)**2)
    return train_mse, test_mse

degrees = list(range(1, 80))
train_errors, test_errors = [], []

for d in degrees:
    tr_mse, te_mse = fit_polynomial_ls(x_train, y_train, x_test, y_test, d)
    train_errors.append(tr_mse)
    test_errors.append(min(te_mse, 50))  # cap for display

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(degrees, train_errors, 'b-o', markersize=3, label='Train MSE')
axes[0].semilogy(degrees, test_errors, 'r-o', markersize=3, label='Test MSE')
axes[0].axvline(n_train - 1, color='green', linestyle='--',
                label=f'Interpolation threshold (d={n_train-1})')
axes[0].set_title('Double Descent: Polynomial Regression')
axes[0].set_xlabel('Polynomial degree'); axes[0].set_ylabel('MSE (log scale)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Zoom on the transition region
axes[1].semilogy(degrees[20:55], test_errors[20:55], 'r-o', markersize=4)
axes[1].axvline(n_train - 1, color='green', linestyle='--',
                label=f'Threshold at d={n_train-1}')
axes[1].set_title('Zoom: Double Descent Peak Region')
axes[1].set_xlabel('Polynomial degree'); axes[1].set_ylabel('Test MSE (log)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s30_09_double_descent.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Interpolation threshold at degree {n_train-1}')
print(f'Peak test error near threshold: {max(test_errors[n_train-5:n_train+5]):.4f}')

In [ ]:
# ── Effect of noise level on double descent peak ──────────────────────────
noise_levels = [0.1, 0.3, 0.5, 1.0]
fig, ax = plt.subplots(figsize=(12, 6))

for noise in noise_levels:
    np.random.seed(42)
    y_noisy = np.sin(2*np.pi*x_train) + noise * np.random.randn(n_train)
    test_errs = []
    for d in degrees:
        _, te = fit_polynomial_ls(x_train, y_noisy, x_test, y_test, d)
        test_errs.append(min(te, 100))
    ax.semilogy(degrees, test_errs, '-o', markersize=2,
                label=f'noise={noise}', linewidth=1.5)

ax.axvline(n_train - 1, color='black', linestyle='--', alpha=0.5,
           label=f'Interpolation threshold')
ax.set_title('Double Descent Peak vs Noise Level')
ax.set_xlabel('Polynomial degree'); ax.set_ylabel('Test MSE (log)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_09_noise_effect.png', dpi=100, bbox_inches='tight')
plt.show()

print('=== Double Descent Key Insights ===')
print()
print('Classical regime (d < n_train):')
print('  Underdetermined: classic bias-variance tradeoff applies')
print()
print('Interpolation threshold (d ~ n_train):')
print('  Model barely fits training data -> ill-conditioned solution -> high test error')
print()
print('Overparameterized regime (d >> n_train):')
print('  Many solutions exist; min-norm solution is smooth -> lower test error')
print('  Implicit regularization from min-norm/gradient descent inductive bias')
print()
print('Practical implication:')
print('  When overfitting occurs near interpolation threshold, go BIGGER not smaller!')